# 03 — Pareto Evaluation across Budget Tiers

Run the extractive summarizer at each budget tier on a held-out subset, compute ROUGE-L and simulated latency, and persist the artifact for the API.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

PROC = Path('../data/processed')
MODELS = Path('../models'); MODELS.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

In [ ]:
test = pd.read_parquet(PROC / 'test.parquet')
print('test rows:', len(test))

## Per-tier evaluation

In [ ]:
from slm_quant.features import BUDGETS
from slm_quant.models import evaluate_budget
rows = [evaluate_budget(test, tier, n_eval=500, seed=42) for tier in BUDGETS]
df = pd.DataFrame(rows)
df

## ROUGE-L vs latency p95

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(df['latency_p95'], df['rouge_l_mean'], s=80, color='steelblue')
for _, r in df.iterrows():
    ax.annotate(f"{r['budget']} ({r['size_mb']} MB)", (r['latency_p95']+1, r['rouge_l_mean']))
ax.set_xlabel('p95 latency (ms)')
ax.set_ylabel('ROUGE-L mean')
ax.set_title('Latency-quality trade-off')
plt.show()

## Pareto frontier

In [ ]:
from slm_quant.models import pareto_frontier
ranked = pareto_frontier(df.to_dict(orient='records'))
df_p = pd.DataFrame(ranked)
fig, ax = plt.subplots(figsize=(7,4))
for _, r in df_p.iterrows():
    color = 'green' if r['on_pareto_frontier'] else 'red'
    ax.scatter(r['latency_p95'], r['rouge_l_mean'], s=80, color=color)
    ax.annotate(f"{r['budget']}", (r['latency_p95']+1, r['rouge_l_mean']))
frontier = df_p[df_p['on_pareto_frontier']].sort_values('latency_p95')
ax.plot(frontier['latency_p95'], frontier['rouge_l_mean'], color='green', linestyle='--')
ax.set_title('Pareto frontier')
ax.set_xlabel('p95 latency (ms)')
ax.set_ylabel('ROUGE-L mean')
plt.show()

## Per-domain ROUGE-L per tier

In [ ]:
long = df.melt(id_vars='budget', value_vars=['rouge_l_news','rouge_l_technical','rouge_l_narrative'],
               var_name='domain', value_name='rouge_l')
long['domain'] = long['domain'].str.replace('rouge_l_', '')
fig, ax = plt.subplots(figsize=(8,3.5))
sns.barplot(data=long, x='budget', y='rouge_l', hue='domain', ax=ax)
ax.set_title('Per-domain ROUGE-L per tier')
plt.show()

## Save artifact

In [ ]:
joblib.dump({'budgets': BUDGETS, 'pareto': ranked}, MODELS / 'summarizer.pkl')
df_p.to_parquet(PROC / 'pareto.parquet', index=False)
df_p

## Latency variance per tier (200 simulated calls)

In [ ]:
from slm_quant.features import estimate_latency
rng = np.random.default_rng(0)
rows_v = []
for tier in BUDGETS:
    for _ in range(200):
        L = int(rng.integers(80, 512))
        rows_v.append({'tier': tier, 'doc_tokens': L, 'latency_ms': estimate_latency(L, tier, rng=rng)})
dfv = pd.DataFrame(rows_v)
fig, ax = plt.subplots(figsize=(7,3))
sns.boxplot(data=dfv, x='tier', y='latency_ms', ax=ax)
ax.set_title('Latency variance per tier (200 simulated calls)')
plt.show()